# GCN - Graph Convolutional Network

We now build our first GNN: a Graph Convolutional Network (GCN).
Unlike the Random Forest baseline, GCN aggregates information from 
neighboring nodes.


In [1]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import EllipticBitcoinDataset
from torch_geometric.nn import GCNConv
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Load dataset
dataset = EllipticBitcoinDataset(root='../data/elliptic')
data = dataset[0]

# Load timesteps
feat_df = pd.read_csv('../data/elliptic/raw/elliptic_txs_features.csv', header=None)
class_df = pd.read_csv('../data/elliptic/raw/elliptic_txs_classes.csv')
feat_df.columns = ['txId'] + [f'f{i}' for i in range(1, feat_df.shape[1])]
feat_df['timestep'] = feat_df['f1'].astype(int)
class_df.columns = ['txId', 'label']
df_meta = feat_df[['txId', 'timestep']].merge(class_df, on='txId')

timesteps = torch.tensor(df_meta['timestep'].values)

print(f"Dataset loaded — Nodes: {data.num_nodes:,} | Features: {data.num_node_features}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

Dataset loaded — Nodes: 203,769 | Features: 165
Device: cuda


## 1. Model Definition

A 3-layer GCN where each layer aggregates features from neighboring nodes.
Dropout is applied between layers to prevent overfitting.

In [2]:
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.conv3 = GCNConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv3(x, edge_index)
        return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GCN(in_channels=165, hidden_channels=64, out_channels=2).to(device)
data = data.to(device)
timesteps = timesteps.to(device)

print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")

GCN(
  (conv1): GCNConv(165, 64)
  (conv2): GCNConv(64, 64)
  (conv3): GCNConv(64, 2)
)

Parameters: 14,914


## 2. Training Setup

We use a temporal split (timesteps 1-34 for training, 35-49 for testing)
and apply class weighting to handle the severe imbalance between licit and illicit nodes.
Only labeled nodes contribute to the loss.

In [3]:
# Masks
labeled_mask = data.y != 2
train_mask = labeled_mask & (timesteps <= 34)
test_mask = labeled_mask & (timesteps > 34)

# Labels: 1=illicit → 1, 2=licit → 0
y_binary = torch.zeros(data.y.shape, dtype=torch.long).to(device)
y_binary[data.y == 1] = 1  # illicit
y_binary[data.y == 2] = 0  # licit

# Class weights
n_licit = (y_binary[train_mask] == 0).sum().item()
n_illicit = (y_binary[train_mask] == 1).sum().item()
weight = torch.tensor([1.0, n_licit / n_illicit]).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = torch.nn.CrossEntropyLoss(weight=weight)

print(f"Train nodes: {train_mask.sum():,}")
print(f"Test nodes:  {test_mask.sum():,}")
print(f"Train illicit: {n_illicit:,}")
print(f"Train licit: {n_licit:,}")
print(f"Class weight illicit: {n_licit/n_illicit:.1f}x")

Train nodes: 29,894
Test nodes:  16,670
Train illicit: 3,462
Train licit: 26,432
Class weight illicit: 7.6x


## 3. Training Loop

In [4]:
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = criterion(out[train_mask], y_binary[train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

def evaluate(mask):
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        pred = out.argmax(dim=1)
        prob = F.softmax(out, dim=1)[:, 1]
    
    y_true = y_binary[mask].cpu().numpy()
    y_pred = pred[mask].cpu().numpy()
    y_prob = prob[mask].cpu().numpy()
    
    auc = roc_auc_score(y_true, y_prob)
    return y_true, y_pred, y_prob, auc

# Training
losses = []
test_aucs = []

for epoch in range(1, 201):
    loss = train()
    losses.append(loss)
    
    if epoch % 10 == 0:
        y_true, y_pred, y_prob, auc = evaluate(test_mask)
        test_aucs.append(auc)
        print(f"Epoch {epoch:3d} | Loss: {loss:.4f} | Test AUC: {auc:.4f}")

Epoch  10 | Loss: 0.4867 | Test AUC: 0.7813
Epoch  20 | Loss: 0.4106 | Test AUC: 0.8088
Epoch  30 | Loss: 0.3631 | Test AUC: 0.7955
Epoch  40 | Loss: 0.3327 | Test AUC: 0.8100
Epoch  50 | Loss: 0.3126 | Test AUC: 0.8118
Epoch  60 | Loss: 0.2927 | Test AUC: 0.8177
Epoch  70 | Loss: 0.2755 | Test AUC: 0.8090
Epoch  80 | Loss: 0.2672 | Test AUC: 0.8216
Epoch  90 | Loss: 0.2477 | Test AUC: 0.8129
Epoch 100 | Loss: 0.2460 | Test AUC: 0.8178
Epoch 110 | Loss: 0.2338 | Test AUC: 0.8136
Epoch 120 | Loss: 0.2256 | Test AUC: 0.8138
Epoch 130 | Loss: 0.2183 | Test AUC: 0.8156
Epoch 140 | Loss: 0.2085 | Test AUC: 0.8124
Epoch 150 | Loss: 0.2052 | Test AUC: 0.8121
Epoch 160 | Loss: 0.1986 | Test AUC: 0.8117
Epoch 170 | Loss: 0.1986 | Test AUC: 0.8101
Epoch 180 | Loss: 0.1876 | Test AUC: 0.8055
Epoch 190 | Loss: 0.1883 | Test AUC: 0.8004
Epoch 200 | Loss: 0.1901 | Test AUC: 0.7879


## 4. Results

The GCN achieves a lower AUC than Random Forest, which is expected,
the node features already include precomputed 1-hop aggregates, 
giving tabular models an unfair advantage on this metric.
The key question is whether GCN improves recall on illicit nodes.

In [5]:
y_true, y_pred, y_prob, auc = evaluate(test_mask)

print("=== GCN Results ===")
print(classification_report(y_true, y_pred, target_names=['Licit', 'Illicit']))
print(f"ROC-AUC: {auc:.4f}")
print(f"\nBaseline RF recall on illicit: 11%")
print(f"GCN recall on illicit: {(y_pred[y_true==1]==1).sum() / (y_true==1).sum() * 100:.0f}%")

=== GCN Results ===
              precision    recall  f1-score   support

       Licit       0.97      0.90      0.93     15587
     Illicit       0.27      0.53      0.35      1083

    accuracy                           0.87     16670
   macro avg       0.62      0.71      0.64     16670
weighted avg       0.92      0.87      0.89     16670

ROC-AUC: 0.7879

Baseline RF recall on illicit: 11%
GCN recall on illicit: 53%
